# Chapter 3 · Reinforcement learning and multi-agent exploration
**Student workbook**

Follow the same section titles and **Task 1 / 2 / 3** labels as the video.

## Session overview

Today we will:

1. Learn **tabular Q-learning** in the Cookie café.
2. Hand-code a **DQN learning step** for highway driving.
3. Use **Stable-Baselines3 DQN** and evaluate driving policies.
4. Try **library MAPPO** and shared-reward coordination in Simple Spread.

Optional self-study: conflicting individual incentives in **Prisoner's Dilemma (Appendix A.4)**.

## 0 · Set up the coding session
Create the environment once using the repository README, then choose the **Python (RL Chapter 3)** kernel.
This notebook checks its environment; it does not install or upgrade packages.
Widgets run in JupyterLab. Ordinary code alternatives accompany the controls.

Run cells in order. In the video, each task starts with blanks, followed by a
solution walkthrough. In this workbook, replace the marked `...` in the editable
function cell, run it, then run its check. Cell tags are organizational labels.
An unfinished blank raises `NotImplementedError`
when called. Required coding is limited to Tasks 1, 2 and 3; multi-agent coding
is optional after the recording ends. There is no solution switch.

Keep the detailed explanations in this workbook as a reference while watching.

In [ ]:
import os, sys, copy, json, time
from pathlib import Path
from importlib.metadata import version

# Find the project from either the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'rl_lab').is_dir() and (p / 'environment.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open JupyterLab from the cloned repository; see README.md.')
sys.path.insert(0, str(ROOT))
import rl_lab  # Sets headless rendering before importing pygame or matplotlib.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display, Image, Markdown, Code
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
from rl_lab.rules import LearningRules
from rl_lab.checks import (check_foundations, check_dqn_targets, check_dqn_loss,
                           check_tit_for_tat, check_team_reward)
from rl_lab.cafe import (CookieCafe, CafeQAgent, cafe_episode, cafe_evaluate,
                         train_cafe, plot_cafe, plot_cafe_policy)
from rl_lab.cafe_reference import solve_cafe_reference
from rl_lab.highway import (HIGHWAY_CONFIG, ACTION_NAMES, make_highway, random_argmax,
                            highway_rollout, highway_evaluate, PlaybackDQN)
from rl_lab.highway_study import plot_highway_learning, plot_highway_outcomes, save_study
from rl_lab.library_dqn import (SB3_DQN_KWARGS, run_sb3_highway_study,
                                load_sb3_study, load_sb3_study_dqn)
from rl_lab.social import (read_payoffs, plot_payoffs, social_match,
                           train_social, plot_social)
from rl_lab.social_reference import load_social_reference
from rl_lab.mappo import load_mappo, REFERENCE as MULTI_AGENT_REFERENCE
from rl_lab.spread import (make_spread, reset_spread, spread_positions,
                            spread_rollout, plot_spread_comparison)
from rl_lab.practice import cafe_controls, dilemma_controls, spread_controls

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(f'Select the Python 3.11 lab kernel. Current Python: {sys.executable}')
torch.set_num_threads(1)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
VERSIONS = {name: version(name) for name in
            ['numpy','torch','stable-baselines3','skrl','gymnasium','highway-env','pettingzoo','mpe2','matplotlib','pandas']}
print('Kernel:', sys.executable)
print('Versions:', VERSIONS)


In [ ]:
# 'smoke': code check; 'teaching': short live demo; 'experiment': full sequential study.
# Default: load the completed SB3 50,000-step / five-seed highway study for the video.
RUN_MODE = os.environ.get('CH3_RUN_MODE', 'teaching')
RUN_HIGHWAY_TRAINING = os.environ.get('CH3_TRAIN_HIGHWAY', '0') == '1'
RUN_SB3_TRAINING = os.environ.get('CH3_TRAIN_SB3', '0') == '1'
RUN_EXTRA_ANALYSIS = False   # Optional figures, manual alternative and older playback.
RUN_OPTIONAL_PRACTICE = False  # Optional Tasks 4 and 5, after the video.
RUN_SOCIAL_TRAINING = False   # Optional fresh IQL run; Appendix A.4 loads the prepared pair.
SHOW_WIDGETS = True
DEVICE = 'cpu'
if RUN_MODE not in {'smoke','teaching','experiment'}:
    raise ValueError('Choose smoke, teaching, or experiment.')
BUDGETS = {
    'smoke': dict(cafe=200, highway=288, eval_n=2, dev_n=2, eval_every=128, seeds=[0], social=20),
    'teaching': dict(cafe=20000, highway=2000, eval_n=10, dev_n=5, eval_every=250, seeds=[0], social=300),
    'experiment': dict(cafe=60000, highway=50000, eval_n=100, dev_n=20, eval_every=1000, seeds=[0,1,2,3,4], social=1500),
}
budget = BUDGETS[RUN_MODE]
OUTPUT_DIR = rl_lab.RESULTS / (RUN_MODE + '_' + time.strftime('%Y%m%d_%H%M%S'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Live-run budget:', budget)
print('Highway:', 'TRAIN SB3 DQN + TABLES (use the live-run budget)' if RUN_HIGHWAY_TRAINING
      else 'LOAD SB3 STUDY (50,000 interactions/policy; five training seeds)')
print('Results directory:', OUTPUT_DIR)


## 1 · Cookie café: learn with a Q-table

![One monochrome robot manages one brown Dubai Chewy Cookie, linked to a brown demand curve with lunch and evening peaks.](../assets/figures/task_cafe.png)

**Read the figure:** One agent sells one product: **Dubai Chewy Cookie**. Brown links the cookie
to its demand curve. The illustration is schematic; the results show actual simulator means.

**Question:** As the owner of this cafe, how many cookies should we prepare before the lunch and evening rushes? Too few means lost
sales; too many means preparation costs and unsold stock. Preparation takes one hour.

| Element | Meaning |
|---|---|
| State | Current hour and cookies in stock |
| Action | Prepare 0, 2, or 4 cookies; action indices are 0, 1, 2 |
| Timing | Serve this hour from current stock; preparation is ready next hour |
| Capacity | At most 8 cookies; reserve space before seeing customers |
| Opening | 10 a.m., with 2 cookies already prepared and charged at cost |
| Prices / costs | Sell for 6 credits; prepare for 3 credits per cookie |
| Reward | Cash profit minus 0.5 credits for each customer we cannot serve |
| Closing | 8 p.m.; dispose of leftovers for 1 credit each; no preparation in the final hour |

**From actions to rewards:** Preparing cookies incurs a cost in the current hour,
while the new cookies become available next hour. They can improve later sales and
reduce later missed-customer penalties; any left at closing incur a waste cost.
Future sales are rewarded when they occur. Learning therefore needs to consider
return across the day, rather than maximizing only the current hour's reward.

**Think before coding:** How can an action lower the current hour's reward but
improve the total reward over the rest of the café day?


### 1.1 · Formalize the café · Rewards and state transitions

Let $s_t=(h_t,b_t)$ contain the hour index and current stock. The action index
$a_t\in\{0,1,2\}$ selects a preparation quantity $u_t\in\{0,2,4\}$; the quantity
is the number of cookies, rather than the action index itself.

**Reward follows the timing of costs and sales.** If demand is $D_t$, sales are
$c_t=\min(b_t,D_t)$ and missed customers are $m_t=D_t-c_t$. Newly prepared cookies
cannot serve this hour's customers. Using the environment's prices and penalties:

$$
r_t=6c_t-3u_t-0.5m_t-C_{\mathrm{open},t}-C_{\mathrm{close},t}.
$$

The opening-stock cost $C_{\mathrm{open},t}$ is 6 credits in the first hour
(two opening cookies at 3 credits each), and zero afterward. The closing cost
$C_{\mathrm{close},t}$ is 1 credit per leftover cookie in the final hour, and
zero otherwise. Thus preparation reduces the immediate reward through its cost,
while its possible sales benefit appears in later rewards. The Q-value estimates
return across these transitions.

**Uncertainty enters the state transition.** The hour advances deterministically;
the amount of stock carried forward depends on random demand:

$$
\begin{aligned}
D_t &\sim \mathrm{Poisson}(\lambda_{h_t}),\\
c_t &= \min(b_t,D_t),\\
s_{t+1} &= \bigl(h_t+1,\; b_t-c_t+u_t\bigr).
\end{aligned}
$$

For a fixed demand realization, this equation determines the next state. Before
seeing demand, several next stock values may be possible from the same state and
action. We describe their probabilities with the **transition model**
$P(s_{t+1}\mid s_t,a_t)$. The same random demand also affects sales and missed
customers, so the realized reward can vary too. The preparation delay is represented
by adding $u_t$ after current sales; it connects today's cost to next hour's stock.

**Only demand is uncertain.** Each hour's customer count is sampled from a Poisson
distribution with an hour-specific mean: a larger peak at **12:00** (4 customers/hour)
and a smaller peak at **18:00** (3 customers/hour), separated by a quiet afternoon.
Selling hours are 10:00–19:00; the café closes at 20:00. These are peaks in
**expected demand**; a particular random day need not have two visible peaks. The agent sees the
hour and stock before choosing, not that hour's arrivals. Ingredients are always
available and preparation always succeeds; there is no remaining-ingredients budget.
The simulator generates each customer day in advance so equal seeds give every policy
the same demand, while keeping future counts hidden from the agent.

Choose how many cookies to prepare before seeing the customers for that hour.
The controls reveal the outcome only after your decision. Try the same seed twice
and explain one change in your plan. Keep this experience in mind for Interpretation.

In [ ]:
if SHOW_WIDGETS:
    cafe_ui = cafe_controls(seed=2026)

### 1.2 · Store action values · NumPy Q-table

The controls showed the consequence of one choice. We now need to compare
choices before seeing the next customers. Immediate reward alone would miss the
benefit of preparing for a later rush, so we compare **expected return**: reward
now plus future rewards. Because hour and stock take only a few discrete values,
a Q-table is a simple way to store an estimate for every state–action pair.
NumPy supplies the storage and indexing; the later cells choose actions and
learn the estimates from experience.

A **Q-value**, `Q(s, a)`, is that expected-return score stored in the table.
Tabular Q-learning improves these estimates through repeated café days. Each state
has two integer indices: **hour and stock**.

- `Q[hour, stock, action]` stores one value per state–action combination.
- `Q[tuple(state)]` selects the three action values for the current hour and stock.
- `.shape` reports each dimension's size. The final dimension represents actions.
- A Boolean **mask** marks feasible actions. `values[mask]` selects their values;
  `np.flatnonzero(mask)` gives their action indices.

Hour index `0` means 10 a.m.; `10` is the terminal state at 8 p.m. after closing. Integer arrays
hold state indices; floating-point arrays hold estimated values.

Inspect **noon with six cookies in stock**. The example Q-values are invented for
indexing practice; a new agent starts with zeros. **Predict:** can we prepare four
more cookies if capacity is eight? Should the largest stored value win?


In [ ]:
numpy_cafe = CookieCafe()
numpy_cafe.t, numpy_cafe.stock = 2, 6  # Noon, six cookies.
cafe_state_demo = numpy_cafe.observation()
cafe_mask_demo = numpy_cafe.action_mask()

# Two state dimensions, followed by one action dimension.
cafe_q_demo = np.zeros((*numpy_cafe.observation_space.nvec,
                       numpy_cafe.action_space.n), dtype=np.float32)
cafe_q_demo[tuple(cafe_state_demo)] = np.array([1., 4., 9.], dtype=np.float32)
cafe_values_demo = cafe_q_demo[tuple(cafe_state_demo)]
print('State [hour index, stock]:', cafe_state_demo, '| dtype:', cafe_state_demo.dtype)
print('Q-table shape:', cafe_q_demo.shape, '| dtype:', cafe_q_demo.dtype)
print('Values for this state:', cafe_values_demo)
display(pd.DataFrame({
    'action_index': np.arange(numpy_cafe.action_space.n),
    'cookies_to_prepare': numpy_cafe.ACTIONS,
    'feasible': cafe_mask_demo, 'example_Q': cafe_values_demo,
}))
print('Feasible action indices:', np.flatnonzero(cafe_mask_demo))
print('Feasible values:', cafe_values_demo[cafe_mask_demo])
print('Best feasible value:', cafe_values_demo[cafe_mask_demo].max())
numpy_cafe.close()
# Try: change current stock above and predict how the mask changes.

### 1.3 · Balance exploration and exploitation · Epsilon-greedy

We have storage and a feasibility mask, but still need useful experience.
A new table starts at zero; later estimates may reflect only a few visits.
Always choosing a current maximum can leave alternatives poorly explored.
Epsilon-greedy keeps trying other feasible actions while also using what we have
learned. The mask handles feasibility; epsilon controls exploration.

`epsilon_greedy` is supplied so Task 1 focuses on learning from a transition.
With probability `epsilon`, select a random feasible action. Otherwise select a
feasible action with the largest Q-value. Break equal-value ties randomly.

`np.flatnonzero(mask)` gives feasible action indices. The mask represents what can
be done, not how good an action is. For example, stock six and capacity eight allow
preparing zero or two cookies, but not four. Both branches must respect that constraint.
Read the supplied function if you want to connect the policy to NumPy indexing.


In [ ]:
def epsilon_greedy(values, epsilon, rng, mask=None):
    values = np.asarray(values, dtype=float)
    mask = np.ones(len(values), dtype=bool) if mask is None else np.asarray(mask, bool)
    valid = np.flatnonzero(mask)
    if not len(valid):
        raise ValueError('No feasible action.')
    if rng.random() < epsilon:
        return int(rng.choice(valid))
    best = valid[values[valid] == values[valid].max()]
    return int(rng.choice(best))


### Task 1 · Learn from TD error · Target and Q-update

The exploration rule chooses an action; the environment supplies reward and
next state. Now improve the stored estimate using that experience. TD learning
combines immediate reward with estimated continuation, connecting preparation
costs to later sales over repeated updates without waiting for a full day's
return at every update.

Complete only **1.1** and **1.2** in the next function cell; the rest is supplied.
An observed transition gives immediate reward and a next state. The TD target
combines that reward with the best feasible estimate of future reward:

$$
y=r+\gamma(1-\mathrm{terminated})\max_{a'\,\mathrm{valid}}Q(s',a'),
\qquad Q(s,a)\leftarrow Q(s,a)+\alpha[y-Q(s,a)].
$$

- **1.1:** use `next_values[next_mask]` to exclude infeasible next actions before
  taking the maximum. That maximum can be negative; do not replace it with zero.
- **1.2:** `alpha` is the fraction of the prediction error used for this update.
  If `alpha=0`, keep the old value. If `alpha=1`, move fully to the target.

True termination has no future value; that branch is already supplied. The café's
closing time is part of the task, with hour included in the state. It uses `gamma=1`
to optimize total daily reward. For a practice calculation, old value 2, target 6
and `alpha=0.5` should give 4.

**Pause the video**, replace the two `...` expressions, run the function cell,
then run the check cell. Expect **“Task 1 passed”**. If a check fails, use its
message to inspect the named behavior and rerun both cells before continuing.


In [ ]:
def td_target(reward, terminated, next_values, next_mask, gamma):
    if terminated:
        return float(reward)
    next_values = np.asarray(next_values)
    next_mask = np.asarray(next_mask, dtype=bool)
    target = ...  # TODO 1.1: reward + discounted maximum over feasible next actions.
    if target is Ellipsis:
        raise NotImplementedError('Task 1.1: fill the target expression.')
    return float(target)

def q_update(old_value, target, alpha):
    updated_value = ...  # TODO 1.2: move old_value an alpha fraction toward target.
    if updated_value is Ellipsis:
        raise NotImplementedError('Task 1.2: fill the update expression.')
    return float(updated_value)


In [ ]:
check_foundations(epsilon_greedy, td_target, q_update)
rules = LearningRules(epsilon_greedy, td_target, q_update)

### 1.4 · Learn from one transition · Update a Q-value

The checks verified your target and update separately. Now connect them to
the simulator: choose with epsilon-greedy, observe a transition, build its target,
and change the entry for the action actually taken. Watch the printed index to
see exactly which estimate this one experience changes.

The agent starts with a zero-filled `Q[hour, stock, action]` array.
The simulator and bookkeeping are in `rl_lab/cafe.py`.


In [ ]:
# One learning transition, with all three student functions visible.
demo_env = CookieCafe()
demo_agent = CafeQAgent(demo_env, rules, seed=0)
state, info = demo_env.reset(seed=0)
action = epsilon_greedy(demo_agent.q[tuple(state)], .2, demo_agent.rng, info['action_mask'])
next_state, reward, terminated, truncated, next_info = demo_env.step(action)
target = td_target(reward, terminated, demo_agent.q[tuple(next_state)], next_info['action_mask'], demo_agent.gamma)
index = (*state, action)
demo_agent.q[index] = q_update(demo_agent.q[index], target, demo_agent.alpha)
print('Updated index:', index, '| target:', target, '| Q:', demo_agent.q[index])
demo_env.close()

### 1.5 · Train and evaluate a policy

One successful update shows the mechanism. A useful policy needs experience
across hours, stock levels and customer days, so repeat the same choose–step–update
cycle. Then test without further learning or exploratory actions on new days.
The heuristic and model-optimal reference help judge how much improvement the
learned decisions achieve.

The customer count is random, while its hourly **mean** is known to the simulator.
We provide a model-optimal reference that maximizes expected daily reward using the
same demand distribution, price, costs, capacity, one-hour preparation delay and closing rules.
It knows the distribution, not the next customers. Its backward-induction implementation
is provided reference material, not another exercise.

Train your Q-table, then evaluate it, a forecast-based heuristic, and the optimal
reference on the same held-out customer days. Knowing the average demand does not
make the heuristic optimal.


In [ ]:
cafe_agent, cafe_history = train_cafe(rules, episodes=budget['cafe'], seed=0)
cafe_reference = solve_cafe_reference()
cafe_eval = {}
cafe_eval_seeds = range(20000, 20000 + (20 if RUN_MODE == 'smoke' else 100))
for label, policy in [('Tabular Q-learning', cafe_agent),
                      ('Forecast-based rule', None),
                      ('Model-optimal reference', cafe_reference)]:
    cafe_eval[label] = cafe_evaluate(policy, cafe_eval_seeds)
cafe_scoreboard = pd.DataFrame({name: frame.mean() for name, frame in cafe_eval.items()}).T
display(cafe_scoreboard.rename(columns={'return_':'return'}).round(2))
print('Model-optimal EXPECTED daily reward:', round(cafe_reference.expected_opening_return, 2))
print('Table above: sampled held-out means; single-day outcomes remain random.')
cafe_example = plot_cafe(cafe_agent, cafe_history, reference=cafe_reference, save_dir=OUTPUT_DIR)
display(Markdown('**Read the plot:** Training reward tracks learning; demand and stock explain shortfalls, while preparation is compared with the optimum at the same states.'))


### Interpretation · Connect delayed rewards to decisions

First follow one real day. Brown identifies cookie quantities. Compare known mean
demand with actual arrivals, then inspect available stock and sales. In the
preparation panel, both policies recommend an action for the SAME hour and stock
observed by the learned agent. Reference recommendations do not change that day's
stock trajectory. Preparation becomes available one hour later.

Does preparation anticipate both the noon peak and the smaller 18:00 peak?
Explain one decision using preparation delay, costs, stockouts or closing waste.
Compare reward, cash profit, missed customers and waste across held-out days,
rather than relying on one successful day.

The model-optimal reference knows the demand distribution, not future customers.
The forecast rule also uses means but is a heuristic. The full hour × stock policy
table, visit counts, and reward-change challenge are optional in Appendix A.2.
There are no new code blanks in this interpretation activity.


## 2 · Learn a value function with DQN · Highway driving

![Line-art highway with only the RL-controlled ego car in brown; every other car and the background are greyscale.](../assets/figures/task_highway.png)

**Read the figure:** 1 marks the vehicle controlled by our RL policy. The grey vehicles form part of its environment.

The café gave us a complete loop: store estimates, explore, observe, and update.
We keep that learning idea, but ask whether the same storage is practical for
continuously varying positions and speeds of surrounding vehicles. What should it do when a slower
car is ahead and another car occupies the next lane?

**Who controls the grey cars?** HighwayEnv supplies their hand-coded driving
rules through `IDMVehicle`: the **Intelligent Driver Model (IDM)** controls
acceleration and braking in response to traffic ahead, and **MOBIL** decides when
to change lane using safety and acceleration-gain criteria. Only the brown car
uses our learning agent; the grey cars do not learn from rewards.

**What is randomized?** At reset, the simulator samples initial lanes, positions
and speeds, plus driver variation. In our installed version, `randomize_behavior()`
samples the velocity exponent `DELTA` in the IDM acceleration formula. These cars then react using their
driving rules, rather than sampling a random action at every step. The same seed
reproduces the starting traffic and driver parameters; later trajectories can
change when our agent acts differently because the other drivers react to the scene.
See [HighwayEnv's behavior models](https://highway-env.farama.org/dynamics/vehicle/behavior/).

The existing `highway-fast-v0` task has five actions: lane left, maintain, lane right, faster, slower.
The return rewards progress/speed and penalizes crashes. A high return alone does not establish safety.

Each observation has **5 vehicles × 5 features = 25 numbers**. Presence identifies real rows;
the remaining features describe position and velocity. Both methods receive the same features.
**Example:** a car is 20 metres ahead, then 21 metres ahead. These are different
observations, but experience in one scene could be useful in the other. Storing
an entry for every exact combination of positions and speeds is impractical.

For a table, we group distances and speeds into ranges called **bins**. Observations
with the same bin assignments use the same entry; finer bins create more combinations
to learn. A **deep Q-network (DQN)** passes the features through one learned function
to predict action values. All observations use the same network parameters, so an
update from one experience can affect predictions for other observations too. This
is what **sharing weights across observations** means. The network can predict at
an unseen observation, but we still need to test whether that prediction is useful.

**Same question, different representation:** the café looks up action values;
the highway network predicts them. Both choose actions using estimated future reward.

**Predict:** What happens to table coverage when you make every bin finer?
Can a network return a value for an observation it has never seen exactly before?


In [ ]:
inspection_env = make_highway(render=True)
sample_observation, _ = inspection_env.reset(seed=0)
display(pd.DataFrame(sample_observation.reshape(5,5),
    columns=['presence','x / 100m','y / lane width','vx / 30m/s','vy / 10m/s']))
fig, ax = plt.subplots(figsize=(10,3))
inspection_frame = inspection_env.render()
ax.imshow(inspection_frame); ax.axis('off')
ax.set_title('HighwayEnv · brown = RL-controlled car; grey = other traffic')
fig.savefig(OUTPUT_DIR / 'highway_scene.png', dpi=180)
plt.show()
display(Markdown('**Read the plot:** The brown car is controlled by the agent; nearby vehicles provide the position and velocity features used to choose an action.'))
print('Input shape:', sample_observation.shape, '| Actions:', ACTION_NAMES)
# Keep one actual transition for the later PyTorch training-update demonstration.
sample_action = 1  # maintain lane and speed
sample_next_observation, sample_reward, sample_terminated, sample_truncated, _ = inspection_env.step(sample_action)
print('Observed reward:', sample_reward, '| terminated:', sample_terminated, '| truncated:', sample_truncated)
inspection_env.close()

### 2.1 · Predict action values · PyTorch Q-network

The highway inspection established the interface: 25 observed features and
five actions. We now replace table lookup with a trainable function mapping those
features to action values. First define the network, then verify its outputs for
one observation and a batch before connecting it to targets and a loss.

**PyTorch tensors** hold observations and action values. PyTorch also calculates
the gradients needed to learn the network's weights.

`QNetwork` below maps **25 input features to five action values**. Its trainable
parameters track gradients automatically. The outputs are estimated returns, so
there is no softmax and they need not sum to one.

Before running, follow the shapes: one observation `(25,)` → one-item batch `(1, 25)`
→ action values `(1, 5)`. A batch of `B` observations gives `(B, 5)` outputs.

| Needed for highway DQN | NumPy form | PyTorch form |
|---|---|---|
| Floating-point observations | `np.array(data, dtype=np.float32)` | `torch.as_tensor(data, dtype=torch.float32)` |
| Inspect dimensions | `x.shape` | `x.shape` |
| Best action value per batch item | `values.max(axis=1)` | `values.max(dim=1).values` |
| Integer action indices | `np.int64` | `torch.long` |

Here `dim=1` is the action dimension, not time. A forward pass only predicts values;
we will inspect the gradient and optimizer update after completing the DQN loss.


In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim=25, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        # (batch, 25) -> (batch, 5); no softmax: these are values, not probabilities.
        return self.net(x)

In [ ]:
torch.manual_seed(0)
shape_demo = QNetwork()  # Untrained: inspect shapes, not driving quality.
observation_tensor = torch.as_tensor(sample_observation, dtype=torch.float32)
one_state_batch = observation_tensor.unsqueeze(0)  # Add a leading batch dimension.
two_state_batch = torch.stack([
    observation_tensor,
    torch.as_tensor(sample_next_observation, dtype=torch.float32),
])
with torch.no_grad():  # Only inspecting predictions; no gradients needed here.
    one_state_values = shape_demo(one_state_batch)
    two_state_values = shape_demo(two_state_batch)
print('Observation tensor:', tuple(observation_tensor.shape), '| dtype:', observation_tensor.dtype)
print('One observation:', tuple(one_state_batch.shape), '→', tuple(one_state_values.shape))
print('Two observations:', tuple(two_state_batch.shape), '→', tuple(two_state_values.shape))
print('Best predicted value per observation:', two_state_values.max(dim=1).values)
# Predict: how would these shapes change for a replay batch of 64 observations?

### 2.2 · Reuse past experience · Replay buffer

The shape demonstration used untrained predictions. Improving them requires
experience from driving. Consecutive observations are closely related, and one
transition can be useful for more than one update. Replay stores transitions so
we can sample batches from past experience. We define storage here; the later
update demonstration puts the saved highway transition into it.

The provided replay buffer stores `(state, action, reward, next_state, terminated)`
transitions. Sampling a batch converts the stored NumPy arrays to PyTorch tensors.
Read the returned shapes: states and next states `(B, 25)`, integer actions `(B,)`,
rewards `(B,)`, and true-termination flags `(B,)`.
The same transitions can be sampled again, allowing the network to reuse experience.


In [ ]:
class ReplayBuffer:
    """Provided storage: keep a bounded history and sample training batches."""
    def __init__(self, capacity=20000, obs_dim=25, seed=0):
        self.states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_states = np.zeros_like(self.states)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.capacity = capacity
        self.position = 0
        self.size = 0
        self.rng = np.random.default_rng(seed)

    def add(self, state, action, reward, next_state, terminated):
        i = self.position
        self.states[i] = state
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_states[i] = next_state
        self.terminated[i] = terminated  # Do not store terminated OR truncated here.
        self.position = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, device):
        indices = self.rng.integers(self.size, size=batch_size)
        arrays = (self.states, self.actions, self.rewards,
                  self.next_states, self.terminated)
        return tuple(torch.as_tensor(array[indices], device=device) for array in arrays)

### Task 2 · Bootstrap future returns · DQN target

Replay records what happened, but we still need a learning target for each
taken action. Immediate reward captures only part of the return, so reuse the
café's TD idea. The online network changes during learning; a separate **target
network**, periodically copied from it, keeps the next-value reference fixed
between copies. The later `learn` method obtains those next values and passes
them into this function.

Fill the single **2** expression. This is the same TD target as Task 1,
calculated for a batch of transitions instead of one table entry:

$$
y_i=r_i+\gamma(1-\mathrm{terminated}_i)\max_{a'}Q_{\theta^-}(s'_i,a').
$$

`rewards`, `terminated` and `next_values` each have shape `(B,)`. Return a target
tensor with shape `(B,)`. `next_values` already contains the target network's
maximum next-action value; you do not need to run a network in this function.
Convert termination flags to floats with `.float()` before arithmetic.

Stop a rollout on termination **or** truncation. Remove the bootstrap term only
on true termination: the highway time limit is an external cutoff. At a crash,
the terminal target is just the observed reward. Unlike the café, time remaining
is not part of the highway task state. The later `learn` method computes targets
inside `torch.no_grad()` so they do not receive gradients.

**Pause**, fill 2, run the function cell, then its check. Expect **“Task 2 passed”**.
For a nonterminal reward 2, next value 4 and `gamma=0.5`, the target should be 4;
the same reward at true termination should give 2.


In [ ]:
def dqn_targets(rewards, terminated, next_values, gamma):
    targets = ...  # TODO 2: one target per transition; mask only true termination.
    if targets is Ellipsis:
        raise NotImplementedError('Task 2: fill the vector target expression.')
    return targets


In [ ]:
check_dqn_targets(dqn_targets)


### Task 3 · Measure prediction error · DQN loss

Task 2 produced a desired value for each sampled transition. Now measure
the difference between that target and the online network's prediction. This
loss will provide the signal for changing weights. Selecting the action actually
taken links each target to the prediction supported by that transition.

Fill **3.1** and **3.2**. The network predicts five action values for each state,
but each stored transition tells us the result of just the action that was taken.
Train those selected outputs toward their corresponding targets.

**How should we measure the gap?** Let $e_i=Q_\theta(s_i,a_i)-y_i$ be the prediction
error for transition $i$. Squared loss gives very large errors a strong influence.
DQN targets are themselves estimates and may be noisy, so we use **Huber loss**
to reduce the influence of unusually large errors. It is quadratic near zero and
linear beyond a threshold. With the threshold set to 1:

$$
\ell(e_i)=
\begin{cases}
\frac{1}{2}e_i^2, & |e_i|\leq 1, \\
|e_i|-\frac{1}{2}, & |e_i|>1.
\end{cases}
\qquad
\mathcal{L}=\frac{1}{B}\sum_{i=1}^{B}\ell(e_i).
$$

For example, error $0.5$ contributes loss $0.125$, while error $4$ contributes
$3.5$. Both receive a penalty, but the larger error is in the linear region.
Huber loss still learns from large errors; its sensitivity to them grows less
than with squared loss. This can help learning stability, without guaranteeing
that a DQN will learn a good policy.

Our code uses PyTorch's `smooth_l1_loss`. Its default `beta=1.0` makes it equal to
Huber loss with threshold 1, and default `reduction='mean'` averages the individual
losses into one scalar. For other thresholds, Smooth L1 and Huber have different
scaling; the equivalence here relies on the default value of 1.
See the [PyTorch Smooth L1 definition](https://docs.pytorch.org/docs/stable/generated/torch.nn.SmoothL1Loss.html).

| Quantity | Shape |
|---|---|
| `q_values` | `(B, 5)` |
| `indices` | `(B, 1)`, integer action indices |
| selected values, after removing dimension 1 | `(B,)` |
| `fixed_targets` | `(B,)`, detached from gradients |
| mean loss | scalar |

For **3.1**, use `gather` along the action dimension (`dim=1`), then `.squeeze(1)`.
Removing that specific dimension preserves the batch dimension even when `B=1`.
For **3.2**, use `nn.functional.smooth_l1_loss` on the selected values and fixed
targets. The supplied `.detach()` prevents target gradients.

Avoid comparing `(B,1)` with `(B,)`: broadcasting can silently create a `(B,B)`
array. For example, selected values `[3, 4]` and targets `[2, 2]` have mean
Huber loss 1.0. Unchosen outputs should have no direct loss gradient in this check.

**Pause**, fill 3.1/3.2, run the function cell, then its check. Expect **“Task 3 passed”**.


In [ ]:
def dqn_loss(q_values, actions, targets):
    indices = actions.long().reshape(-1, 1)
    chosen_q = ...  # TODO 3.1: gather chosen actions and remove only dimension 1.
    if chosen_q is Ellipsis:
        raise NotImplementedError('Task 3.1: select one action value per transition.')
    fixed_targets = targets.detach()
    loss = ...  # TODO 3.2: mean smooth_l1_loss(chosen_q, fixed_targets).
    if loss is Ellipsis:
        raise NotImplementedError('Task 3.2: calculate the scalar loss.')
    return loss


In [ ]:
check_dqn_loss(dqn_loss)


### 2.3 · Update network weights · One DQN gradient step

Calculating targets and loss measures prediction error. Changing future
predictions requires an optimizer update. We now connect the network, replay,
Task 2 targets and Task 3 loss inside `DQNAgent.learn`. After defining the agent,
use the real transition saved during highway inspection and check which weights
changed. This parallels the café's one-transition demonstration.

The café changed one table entry directly. DQN changes network weights so its
chosen action values move toward targets built from observed transitions.
The online network receives gradients; the target network supplies fixed targets
and is periodically copied from the online network.

Follow the actual `learn` method below:

1. Compute next-state targets inside `torch.no_grad()`; targets should not receive gradients.
2. Calculate a scalar loss from the online network's predictions and those targets.
3. `optimizer.zero_grad()` clears gradients left from the previous update.
4. `loss.backward()` calculates how each trainable parameter affects the loss.
5. `optimizer.step()` uses those gradients to change the online weights.

**Predict:** If we compute a loss and call `backward()` but omit `optimizer.step()`,
will the driving policy's weights change? How is that different from `sync_target()`?


In [ ]:
class DQNAgent:
    def __init__(self, seed=0, gamma=.95, learning_rate=1e-3, device='cpu'):
        torch.manual_seed(seed)
        self.device = torch.device(device)
        self.online = QNetwork().to(self.device)
        self.target = copy.deepcopy(self.online).to(self.device)
        self.target.eval()
        for parameter in self.target.parameters():
            parameter.requires_grad_(False)
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=learning_rate)
        self.buffer = ReplayBuffer(seed=seed + 17)
        self.rng = np.random.default_rng(seed)
        self.gamma = gamma
        self.last_loss = np.nan

    def act(self, state, epsilon=0.):
        if self.rng.random() < epsilon:
            return int(self.rng.integers(5))
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=self.device)
            values = self.online(state_tensor).cpu().numpy()
        return random_argmax(values, self.rng)

    def learn(self, batch_size=64):
        states, actions, rewards, next_states, terminated = self.buffer.sample(batch_size, self.device)
        # Fixed targets: gradients must not flow through the target network.
        with torch.no_grad():
            next_values = self.target(next_states).max(dim=1).values
            targets = dqn_targets(rewards, terminated, next_values, self.gamma)

        loss = dqn_loss(self.online(states), actions, targets)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), 10.)
        self.optimizer.step()
        self.last_loss = float(loss.detach().cpu())
        return self.last_loss

    def sync_target(self):
        self.target.load_state_dict(self.online.state_dict())

In [ ]:
# One real driving transition, one gradient update; this is not a trained driving policy.
update_demo = DQNAgent(seed=0)
update_demo.buffer.add(sample_observation, sample_action, sample_reward,
                       sample_next_observation, sample_terminated)
online_before = {name: value.detach().clone()
                 for name, value in update_demo.online.state_dict().items()}
target_before = {name: value.detach().clone()
                 for name, value in update_demo.target.state_dict().items()}
example_loss = update_demo.learn(batch_size=1)
print('Loss from the observed transition:', example_loss)
print('First layer gradient norm:', next(update_demo.online.parameters()).grad.norm().item())
print('Online weights changed:', any(
    not torch.equal(value, online_before[name])
    for name, value in update_demo.online.state_dict().items()))
print('Target weights unchanged:', all(
    torch.equal(value, target_before[name])
    for name, value in update_demo.target.state_dict().items()))

### 2.4 · Build or reuse an agent · Stable-Baselines3 DQN

A complete experiment must repeat collection and updates, refresh targets,
manage episode boundaries and record results. Having inspected one update, we
can use a standard implementation for that repeated training process. The next
cell demonstrates the SB3 API; the following section evaluates separately trained
SB3 policies.

**Stable-Baselines3 (SB3)** supplies the replay buffer, target network,
exploration schedule and training loop for the experiment.

| When | What to use |
|---|---|
| Understand, debug or teach a learning rule | Hand-code a small update you can inspect |
| Establish a baseline or run a standard algorithm | Use an existing implementation such as SB3 |
| Change the network or add logging/evaluation | First try SB3 policy settings and callbacks |
| Research a learning rule that the library cannot express | Implement the necessary custom agent/update and test it |

Writing every agent from scratch is not a requirement for doing RL research.
Using a library still requires choosing the state, actions, reward, training budget
and evaluation. Our lesson uses both approaches: **hand-code to understand, then
use the library for the highway experiment**. The table-based café remains our
earlier hand-coded value-learning example.

| Hand-coded part | SB3 counterpart |
|---|---|
| `QNetwork` | `MlpPolicy`, with `policy_kwargs` for its architecture |
| `ReplayBuffer` | Storage configured through `buffer_size` |
| Targets, loss and optimizer in `DQNAgent.learn` | Updates performed by `.learn(...)` |
| `act` for evaluation | `.predict(observation, deterministic=True)` |

The next cell uses the same `make_highway()` wrapper and its 25 input features.
Constructing the model or predicting once does not train it. The API demonstration
is explicitly **untrained** by default; `RUN_SB3_TRAINING=True` optionally enables
a short `.learn()` call here. The following experiment creates fresh SB3 models
for each training seed, or loads the separately recorded SB3 study.

`SB3_DQN_KWARGS` fixes the experiment settings before training: a 64–64 MLP,
replay, target updates and exploration are all supplied by SB3. The experiment's
recorded curves and GIF come from SB3, not from the one-update hand-coded demo.
The previous hand-coded study is retained separately as historical material.

See the [SB3 DQN API](https://stable-baselines3.readthedocs.io/en/v2.7.1/modules/dqn.html)
and [callbacks](https://stable-baselines3.readthedocs.io/en/v2.7.1/guide/callbacks.html).


In [ ]:
from stable_baselines3 import DQN as LibraryDQN

sb3_env = make_highway()
try:
    library_agent = LibraryDQN('MlpPolicy', sb3_env, **SB3_DQN_KWARGS,
                              seed=0, device=DEVICE)
    if RUN_SB3_TRAINING:
        library_agent.learn(total_timesteps=budget['highway'])
    observation, _ = sb3_env.reset(seed=2026)
    action, _ = library_agent.predict(observation, deterministic=True)
    sb3_status = (f"trained for {library_agent.num_timesteps} steps"
                  if RUN_SB3_TRAINING else 'UNTRAINED API demonstration')
    print('Stable-Baselines3:', sb3_status)
    print('Predicted action:', ACTION_NAMES[int(action)])
finally:
    sb3_env.close()


### 2.5 · Compare learned policies · SB3 DQN and Q-tables

Creating an agent and calling `predict` demonstrates the package interface.
Assessing learning requires trained policies and held-out evaluation, just as
after the café's single update. We now load actual training runs, compare their
outcomes with tabular agents, and inspect one driving episode after reading the
aggregate scores.

The DQN in this experiment is **`stable_baselines3.DQN`**. The hand-coded
`DQNAgent` above is used only to understand and inspect the update.

The recorded comparison uses **50,000 interactions per policy and five training
seeds**. We trained five new SB3 DQN models and retained verified tabular and
non-learning baseline measurements from the same environment, observations,
actions, budgets and evaluation scenarios. The new manifest identifies every source.
SB3 settings were fixed before collection; this was not a new equal-budget tuning study.

Start with the grey dashed medium-bin table and brown SB3 DQN. Training contains
exploration; development evaluation uses the current greedy policy. The plotted
training and evaluation curves use trailing means over 50 completed episodes and
five checkpoints. The band is ±1 SD across smoothed training runs, not a confidence
interval. Smoothing can hide short dips; Appendix A.3 explains the raw records.

Final-score dots are each trained policy's mean on the same 100 new traffic scenarios.
Read reward, crashes and duration together. One animation shows behavior, not
reliability. Set `RUN_HIGHWAY_TRAINING=True` and rerun in order to train new tables
and **SB3 DQN** using the selected mode; a 2,000-step teaching run is only a short
demonstration and should not replace the full study's measurements.


In [ ]:
# SB3 settings are fixed in advance; table alpha comes from the earlier tabular pilot.
if RUN_HIGHWAY_TRAINING:
    highway_study = run_sb3_highway_study(
        rules, steps=budget['highway'], seeds=budget['seeds'],
        eval_every=budget['eval_every'], eval_seeds=range(82000, 82000 + budget['dev_n']),
        test_seeds=range(93000, 93000 + budget['eval_n']), device=DEVICE,
        dqn_kwargs=SB3_DQN_KWARGS, table_kwargs={'alpha': 0.3})
else:
    highway_study = load_sb3_study()
highway_source = highway_study.metadata['source']
highway_agents = highway_study.agents
print('DQN implementation:', highway_study.metadata['dqn_implementation'])
print('Source:', highway_source)
print('Measured budget:', highway_study.metadata['steps_per_method'],
      'interactions/policy | training seeds:', highway_study.metadata['training_seeds'])
highway_plot_windows = dict(training_window=50, evaluation_window=5)
highway_training_smoothed, highway_evaluation_smoothed = plot_highway_learning(
    highway_study, save_dir=OUTPUT_DIR, **highway_plot_windows)
display(Markdown('**Read the plot:** Training includes exploration; greedy evaluation tests the current policy, with shading (when shown) indicating variation across training seeds.'))
highway_summary = plot_highway_outcomes(highway_study, save_dir=OUTPUT_DIR)
display(Markdown('**Read the plot:** Compare reward with crashes and duration; dots show run means for learned agents and one evaluation mean for each fixed baseline.'))


In [ ]:
# Seed 0 is fixed in advance. This is the actual SB3 study model.
shown_seed = budget['seeds'][0] if highway_agents else 0
shown_dqn = highway_agents[('DQN', shown_seed)] if highway_agents else load_sb3_study_dqn(seed=shown_seed)
print('SB3 DQN:', highway_study.metadata['steps_per_method'],
      'interactions | training seed:', shown_seed)
current_rollout = highway_rollout(shown_dqn, seed=2026, save_path=OUTPUT_DIR / 'current_sb3_dqn.gif')
print('Shown episode return:', current_rollout.reward.sum(), '| Crash:', current_rollout.crashed.any())
display(Markdown('**Read the plot:** One seeded episode illustrates the policy’s actions; use the held-out scores to judge reliability.'))


**Interpret:** Compare medium-bin tabular Q-learning with SB3 DQN using reward,
crashes and the five final-score dots. Why can exploratory training and greedy
evaluation differ? Why does a library agent still need held-out evaluation?
Write two sentences using the displayed measurements. The earlier hand-coded
DQN scores are a different implementation and are not the SB3 result.

## 3 · Explore shared rewards and coordination · Simple Spread

![Monochrome robots and landmarks with blue coverage paths: 1 duplicate nearest-target choices, 2 different assignments, 3 one identical team reward.](../assets/figures/task_simple_spread.png)

**Why move from highway driving to this multi-agent environment?** Highway trained
one decision-maker while other traffic followed supplied simulator rules. Now
several agents learn, and their combined moves determine coverage. Simple Spread
gives all agents the **same team reward**: how do individual decisions combine
into coordinated behavior? Shared reward also creates a credit-assignment problem:
which agent's action caused the team's improvement?

**A different kind of multi-agent problem:** In Prisoner's Dilemma, players have
separate payoffs and potentially conflicting individual incentives. It combines
competition with the possibility of mutual cooperation; it is not a purely
zero-sum contest. Explore its payoff plot, tit-for-tat and Independent Q-learning
in **Appendix A.4** after the video. Here we focus on cooperative coordination.

**Read the figure:** 1 shows duplicated coverage; 2 shows complementary assignments; 3 shows the same reward for everyone. A, B and C correspond to agent_0, agent_1 and agent_2.

Three movable agents must cover three fixed landmarks. If everyone follows their own
nearest landmark, they can duplicate each other's work and leave other landmarks unattended.
There are no preassigned jobs. Agents need complementary behavior to cover the whole scene.

| Element | Meaning |
|---|---|
| Observe | Own velocity and position, relative landmark and teammate positions; 18 values per agent |
| Act | `0` stay, `1` left, `2` right, `3` down, `4` up; all agents choose before the world advances |
| Receive | Every agent receives **the same team coverage reward** at every step |
| Configuration | Actual MPE2 Simple Spread, `N=3`, `local_ratio=0.0`, discrete actions, 60-step horizon |

The four communication entries in each observation are zero: these agents are silent.
Physical collisions remain, but there is **no individual collision penalty**, private
distance reward, or individual bonus. Zero is the best possible coverage reward;
feedback improves as distances shrink, even before all landmarks are covered.

### 3.1 · Calculate a shared reward · NumPy distances

Before comparing controllers, make the objective precise. Every landmark needs
coverage, so the outer sum runs over landmarks. For each **landmark**, find its
nearest agent, sum those distances and negate. The next two cells implement this
calculation and verify it against every agent's actual simulator reward:

$$
r_t=-\sum_{\ell=1}^{3}\min_{i\in\{0,1,2\}}\|x_{i,t}-L_\ell\|_2,
\qquad r_{0,t}=r_{1,t}=r_{2,t}=r_t.
$$

Here $x_{i,t}$ is agent $i$'s position at step $t$, $L_\ell$ is fixed landmark
$\ell$'s position, and $\|\cdot\|_2$ is Euclidean distance. The minimum finds the
closest of agents 0, 1 and 2 for each landmark. The outer sum adds the distances
for landmarks 1, 2 and 3. Every agent receives that same scalar $r_t$.

**Why are the rewards negative?** Distances cannot be negative, so negating their
sum gives $r_t\leq 0$. If the nearest-agent distances are $0.1$, $0.2$ and $0.3$,
the reward is $-(0.1+0.2+0.3)=-0.6$. As coverage improves, reward moves toward its
theoretical maximum of zero, reached when every landmark has an agent exactly
at its position.

**Step reward, return and mean return:** The plotted curve shows the shared reward
at each step. The table's episode return sums it over the full 60-step rollout:

$$
G=\sum_{t=1}^{60}r_t.
$$

We count the shared reward once per step. Mean return averages these episode
totals across the evaluation starts. For example, a constant reward of $-0.5$
for 60 steps gives return $-30$. Travelling to landmarks incurs negative rewards,
even if the agents finish with excellent coverage. Consequently, a negative
return is expected: $-33.90$ is better than $-86.22$. The reward is a continuous
distance measure; counting landmarks within 0.2 world units is a separate
evaluation diagnostic, not an extra training reward.

Return one scalar. Inputs have shape `(n_agents, 2)` and `(n_landmarks, 2)`.
The distance matrix should have shape `(n_agents, n_landmarks)`.
**Hint:** broadcast with `[:, None, :]` and `[None, :, :]`; take a norm over coordinates,
then a minimum **over agents**, not over landmarks.

Before coding: with landmarks at `(0,0)` and `(10,0)`, what happens when agents at
`(1,0)` and `(2,0)` both move left? How would sending one agent right change team coverage?

The function is supplied for the video. Optional Task 5 in Appendix A.5 lets
you fill the distance-reduction expression afterward. Reward sharing raises a
credit-assignment question: the score improves for every agent even if only one
agent caused the improvement.


In [ ]:
def team_reward(agent_positions, landmark_positions):
    distances = np.linalg.norm(
        agent_positions[:, None, :] - landmark_positions[None, :, :], axis=-1)
    return float(-distances.min(axis=0).sum())

In [ ]:
check_team_reward(team_reward)

# One real parallel environment step: all decisions are submitted together.
spread_env = make_spread()
try:
    observations, _ = reset_spread(spread_env, seed=2026)
    print('Observation shapes:', {name: obs.shape for name, obs in observations.items()})
    actions = {name: 0 for name in spread_env.agents}  # everyone stays for this inspection
    next_observations, rewards, terminated, truncated, infos = spread_env.step(actions)
    agent_positions, landmark_positions = spread_positions(spread_env)
    calculated = team_reward(agent_positions, landmark_positions)
    np.testing.assert_allclose(list(rewards.values()), calculated)
    print('Actual environment rewards:', dict(rewards))
    print('Your calculated team reward:', calculated)
finally:
    spread_env.close()

### 3.2 · Learn coordinated behavior · Library MAPPO

The reward is now clear, but giving everybody the same number does not tell them
which moves will cover different landmarks. First try **nearest-landmark steering**:
several agents can head for the same target. We then ask whether a trained policy
has learned complementary moves.

**What is the clustered demonstration?** It is the fixed teaching start used in
the next comparison: all three agents begin near landmark L1, while L2 and L3 are
farther away. The positions stay fixed when this layout is reset. Because all three
initially have the same nearest landmark, this scene makes duplicated effort easy
to see. We deliberately practiced this layout in some training episodes; it is an
illustration, not an unseen test. Both controllers begin from exactly the same
positions. To try new layouts, set `layout='random'` with the same seed in all
rollout calls; the table separately evaluates 50 reserved random starts.

**Why MAPPO here?** **Multi-Agent Proximal Policy Optimization (MAPPO)** is used for
cooperative multi-agent reinforcement learning: several agents learn to act toward
a common team objective. It fits this example because coverage depends on the
three agents' combined positions and they receive the same reward. It extends PPO,
a policy-gradient method that learns action probabilities with a clipped
objective to discourage overly large policy updates. Here an **actor**
network maps an agent's own 18-value observation to probabilities for its five
moves. During training, a **critic** estimates future team return using the
54-value global environment state. That broader context helps evaluate experience
while the actors learn. At execution each actor uses only its own observation;
we do not call the critic or a central action planner. This is **centralized
training with decentralized execution**. In this environment local observations
already contain relative teammate and landmark positions; decentralized execution
does not mean that agents are blind to one another. The agents have the same observation/action formats, so we share one actor
and one centralized critic across them.

**What does “learn from each other's experience” mean here?** During training,
each agent contributes records of its own observations, chosen actions and the
subsequent team rewards. The library estimates advantages using the centralized
critic and combines the agents' current rollout records to update the **same actor
network**. For example, agent 0's experience approaching an uncovered landmark
can improve how that shared network responds to a similar situation. Agents 1
and 2 then use the updated network too. They pool training examples; they do not
send each other messages or copy one another's actions. Unlike DQN's replay buffer,
MAPPO uses freshly collected rollouts for these updates.

At execution, the actor is called separately with each agent's own observation.
The same parameters can produce different moves because the input observations
differ. This parameter sharing is a choice for these interchangeable agents;
sharing rewards alone does not require sharing network parameters. During the
recording comparison, learning is frozen: the agents use what was learned offline
and do not update from one another's current moves.

We use the actual `skrl.multi_agents.torch.mappo.MAPPO` implementation, pinned to
**skrl 1.4.3**. The library handles rollout storage, advantage estimates, clipped
PPO losses and optimization. Our adapter supplies the networks and descriptive
inputs, configures training, and handles loading and simulator interaction. The NumPy function above specifies the environment's
reward; it is not an implementation of the MAPPO learner. We do not derive or
hand-code PPO in this introductory session.

The next cell loads the selected genuine library checkpoint; the printed output
reports its training-run budget and checkpoint step. We train offline, so no
training or internet access is needed during recording. A joint step is one
simultaneous decision by all three agents in one environment. Sixteen parallel
environments provide batches; all agents' experience updates the same policy.

**What improved this run?** The actor appends six descriptive distance features to
its 18 local observation values: its own distance to each landmark and the closest
teammate's distance to each landmark. These distances are calculated entirely from
positions already in that agent's observation. They supply no hidden global state,
assigned targets or expert actions. The library learns all five action probabilities.
The critic likewise derives distance features from its 54-value global input.
These adapter details are provided, not another coding task. The library bootstraps
at the simulator's time-limit truncation, following the Task 2 distinction. All
printed rewards use the original shared-reward units; Appendix A.6 explains the
network and numerical settings used in training.

Training starts are 80% random and 20% the fixed clustered teaching scene. We choose
checkpoints using mean return on 16 development starts, then evaluate the selected
policy once on 50 reserved random starts. Training seeds occupy a separate numeric
range. The clustered scene was practiced during training: it illustrates behavior,
while the new random starts check generalization. Training and candidate selection
are recorded in the manifests; the old checkpoint remains separately archived.

At execution we choose the most probable categorical action. We compare the learned
policy and fixed baselines from identical starts, using the same 60-step horizon.
The final scenes, reward curves and held-out table below show the actual simulator
outcomes. This selected run covers all three landmarks in the practiced teaching
scene. On 50 new random starts, its mean team return is −33.90, compared with
−64.73 for nearest steering and −21.29 for fixed centralized assignment. Mean
final coverage is 2.70 of 3 landmarks, counting a landmark as covered when an agent
is within 0.2 world units; all three are covered in 41 of the 50 starts. This
coverage count is an evaluation measure, not a new training reward. The archived
previous MAPPO model scores −86.22 on these same 50 starts, evaluated only after
selection was fixed. Three development configurations were tried, each with a
3,000,000-joint-step budget; the chosen configuration's checkpoint at 2,400,960
steps had the best development return. One training seed does not measure
variability across training runs or establish a universal ranking. `mappo_metadata` records the selected checkpoint,
settings, development/test seeds, versions and hashes. See README for reproduction.

The controls also include **centralized assignment (fixed)**: a hand-coded planner
uses all starting positions to assign different landmarks, then agents steer to
those targets. This is a coordination reference, not a learned MAPPO policy.

**Why keep credit assignment?** One agent can improve coverage and raise everybody's
reward, including an agent that did nothing. MAPPO uses team context and return
estimates to guide learning, but it does not output a separate causal contribution
for each agent. A centralized critic helps learning; shared reward alone still
does not identify who caused an improvement.

Sources: [skrl MAPPO implementation, pinned release](https://github.com/Toni-SM/skrl/blob/1.4.3/skrl/multi_agents/torch/mappo/mappo.py),
[MAPPO paper](https://arxiv.org/abs/2103.01955).

In [ ]:
# This is the installed library class; the adapter handles model/loading details.
from skrl.multi_agents.torch.mappo import MAPPO
mappo_policy, mappo_metadata = load_mappo()
assert isinstance(mappo_policy.agent, MAPPO)
print(f"MAPPO: {mappo_metadata['library']} {mappo_metadata['library_version']}; "
      f"{mappo_metadata['joint_steps']:,} offline joint training steps; "
      f"loaded checkpoint at {mappo_metadata.get('selected_joint_steps', mappo_metadata['joint_steps']):,} steps; actors only at evaluation")
nearest_rollout = spread_rollout('nearest', seed=2026)
mappo_rollout = spread_rollout('mappo', seed=2026, policy=mappo_policy)
assigned_rollout = spread_rollout('assignment', seed=2026)  # Fixed coordination reference in the controls.
plot_spread_comparison(nearest_rollout, mappo_rollout,
                       save_path=OUTPUT_DIR / 'simple_spread_comparison.png')
display(Markdown('**Read the plots:** MAPPO covers three different landmarks while nearest-landmark steering chases L1; its shared reward stays closer to zero.'))
spread_scoreboard = pd.DataFrame([
    dict(controller=label, team_return=r['history'].team_reward.sum(),
         final_reward=r['rewards'][-1],
         identical_agent_rewards=all(np.allclose(r['history'][f'reward_agent_{i}'], r['history'].team_reward)
                                     for i in range(3)))
    for label, r in [('Nearest (fixed)', nearest_rollout), ('MAPPO (skrl, learned)', mappo_rollout),
                     ('Assignment (fixed, centralized)', assigned_rollout)]])
display(spread_scoreboard)
print('Team return counts the shared reward once per step, not three times.')
spread_test_results = pd.read_csv(MULTI_AGENT_REFERENCE / 'spread_evaluation.csv')
print(f'Held-out random starts ({spread_test_results.seed.nunique()} matched seeds, one MAPPO training run):')
display(spread_test_results.groupby('controller')[['team_return', 'final_reward', 'landmarks_within_02']].mean().round(2))
display(Markdown('**Read the table:** MAPPO improves on nearest steering across 50 new starts, averaging 2.70 of 3 landmarks covered (within 0.2 world units at the end); fixed centralized assignment remains stronger.'))
if SHOW_WIDGETS:
    spread_ui = spread_controls(team_reward, nearest_rollout, assigned_rollout, learned=mappo_rollout)

**Ordinary-code alternative:** the comparison runs without widgets. Change `seed`
and `layout='random'` in all three rollout calls for another matched start; pass
`policy=mappo_policy` for the learned rollout. The slider replays actual positions
and rewards captured from the simulator, without retraining.

**Explain:** Why can moving away from your own nearest landmark help the team?
If agent 0 stays still while agent 1 improves coverage, why do both receive a
better reward? What information does the critic get during training that is not
passed to the actor? Does a shared reward require identical observations, actions
or network weights? MAPPO can learn from team outcomes; its centralized critic
does not by itself solve causal credit assignment. Compare the single teaching
trajectory with the held-out evaluation before claiming reliable coordination.

## 4 · Connect value learning and multi-agent decisions

You completed three coding tasks: a tabular target/update, a vector DQN target,
and an action-value loss. A table stores an estimate for each state-action entry;
a Q-network can use what it learned from one observation when predicting another.
Both improve future-reward estimates from experience. We hand-coded the DQN update
to understand it, then used SB3 for the driving experiment.

In Simple Spread, library MAPPO used team information during training and local
observations for actors at execution. Nearest steering and centralized assignment
stayed fixed references. Shared rewards connected coordination to credit assignment:
everybody can benefit without the reward identifying whose action caused it.

**Exit questions:**
1. Why did a Q-table fit the café, and what motivated the highway network?
2. What do the target and loss contribute to a DQN update?
3. Why distinguish training return, held-out evaluation and a single demonstration?
4. Why doesn't a shared reward identify each agent's contribution to coordination?

The video ends here. The appendix has optional activities and reference material.
Only Tasks 1/2/3 are required. MAPPO is a provided example. For a different kind
of multi-agent problem, Appendix A.4 explores conflicting individual incentives
in Prisoner's Dilemma, with tit-for-tat and two Independent Q-learners for self-study.

## Appendix · Extend and explore

The following sections are outside the 30-minute video. Core functions remain
available even when these activities are skipped.

- **A.4 Prisoner's Dilemma:** run its supplied payoff, tit-for-tat and frozen
  Independent Q-learning cells for self-study; no extra flag is required.
- `RUN_EXTRA_ANALYSIS`: manual café alternative, full café policy table and older DQN playback.
- `RUN_OPTIONAL_PRACTICE`: separate tit-for-tat and team-reward exercises.
- `RUN_SOCIAL_TRAINING`: independent Prisoner's Dilemma Q-learners.
- `RUN_HIGHWAY_TRAINING`: retrain the tables and SB3 DQN; restart from setup.
- `RUN_SB3_TRAINING`: train the short SB3 API example; restart from setup.

For an optional activity, change its flag in setup and rerun that setup cell before
running the activity's cells. Setting a flag does not execute the associated cells.
If you change run mode or training flags, restart and rerun in order.


### A.1 · Test your own decisions · Manual café policy

**Ordinary-code alternative:** edit the ten action indices below and rerun.
An infeasible choice becomes waiting in this manual demo, and is explicitly reported.
The simulator itself rejects infeasible actions.


In [ ]:
if RUN_EXTRA_ANALYSIS:
    MANUAL_ACTIONS = [1, 1, 1, 0, 0, 0, 1, 1, 0, 0]  # Indices: 0→0, 1→2, 2→4 cookies.
    manual_env = CookieCafe()
    invalid_requests = []
    def manual_policy(state, mask):
        requested = MANUAL_ACTIONS[int(state[0])]
        if requested not in range(len(mask)) or not mask[requested]:
            invalid_requests.append((10 + int(state[0]), requested))
            return 0
        return int(requested)
    manual_return, manual_rows = cafe_episode(manual_env, manual_policy, seed=2026)
    display(pd.DataFrame([{
        'hour': r['hour'], 'stock': r['inventory'], 'prepare_next': r['produced'],
        'demand': r['demand'], 'sales': r['sales'], 'cash_profit': r['cash_profit']
    } for r in manual_rows]))
    print('Return:', manual_return, '| Requests replaced by waiting:', invalid_requests)
    manual_env.close()


### A.2 · Inspect state coverage · Café policy maps

**First follow one real day.** Brown identifies cookie quantities. Compare the
known mean demand with the day's actual arrivals, then inspect available stock and sales.
In the preparation panel, both policies recommend an action for the **same hour and
stock observed by the learned agent**. The reference recommendations do not change
that day's stock trajectory; they show what the reference would choose at those states.
Every preparation decision becomes available one hour later.

**Then read the full policy table.** Each cell asks: *at this hour, with this many
cookies available, how many should I prepare?* Rows show stock and columns show
hour. There is no fixed ingredient condition or second-product stock hidden in the plot.

- A number is the quantity to prepare: `0`, `2`, or `4`. Here `0` really means wait.
- `?` in the learned table means that state received no training update. It does
  **not** mean actions are forbidden. With a fixed opening stock of 2, some hypothetical
  opening states are never encountered; finite training can also miss rare states.
- The model-optimal table answers the hypothetical question for every hour and stock,
  including states not reached from our fixed opening. It does not require training visits.

Does preparation anticipate both the noon peak and the smaller 18:00 peak? Choose one visited cell and compare
the learned quantity with the reference. Explain one difference using preparation
delay, cost, stockouts or closing waste. Compare reward, cash profit, missed customers
and waste rather than relying on one successful day.

**Optional inspection:** `cafe_policy_comparison` records hour, stock, visits, preparation
quantities and the model-based action-value gap. A zero gap includes equally optimal ties.

**Optional experiment:** increase `missed_penalty` to 2.0, retrain, and compare profit and missed customers.
Do not directly compare raw rewards from two different reward definitions.

Set `RUN_EXTRA_ANALYSIS=True` to display the full table below.


In [ ]:
cafe_policy_comparison = None
if RUN_EXTRA_ANALYSIS:
    cafe_policy_comparison = plot_cafe_policy(cafe_agent, reference=cafe_reference,
                                             save_dir=OUTPUT_DIR)
    display(Markdown('**Read the plot:** Each number is cookies to prepare at that hour and stock; “?” marks a state without a training update.'))


In [ ]:
RUN_CAFE_REWARD_CHALLENGE = False
if RUN_CAFE_REWARD_CHALLENGE:
    changed_agent, _ = train_cafe(rules, episodes=budget['cafe'], seed=0,
                                 env_kwargs={'missed_penalty': 2.0})
    changed_scores = cafe_evaluate(changed_agent, range(20000,20100),
                                   env_kwargs={'missed_penalty':2.0})
    display(pd.DataFrame({'original':cafe_eval['Tabular Q-learning'].mean(),
                          'retrained_service_priority':changed_scores.mean()}).T[['profit','missed','waste']])

### A.3 · Design a reproducible comparison · Highway study protocol

The experiment uses `stable_baselines3.DQN` 2.7.1 with fixed settings from
`SB3_DQN_KWARGS`: 64–64 hidden layers, learning rate 0.001, discount 0.95,
20,000 replay capacity, 256 warm-up steps, batches of 64, one gradient update per
four interactions, target updates every 400 interactions, and epsilon declining
from 1 to 0.05 over the first 70% of training.

Each learned policy has 50,000 interactions and training seeds 0–4. Development
evaluation uses traffic seeds 82000–82019 every 1,000 interactions; final policies
are evaluated on disjoint seeds 93000–93099. SB3's intermediate step callbacks run
before the associated optimizer update; the final checkpoint is evaluated after
the final update. The final policy is selected by budget, not by test score.
Evaluation uses a separate environment and restores global RNG and policy mode.
SB3's replay buffer handles the difference between true termination and truncation.

The table/baseline records are reused from the verified earlier study, with the
same features, simulator, budgets and development/test scenarios. Tabular alpha
0.3 came from its earlier development pilot. SB3 settings were fixed before the new
collection; this is not a claim of equal hyperparameter-search effort. The new
`assets/reference_results/highway_sb3_study/manifest.json` records SB3 versions,
source hashes, reused records and actual `.zip` model files. The original
`highway_study/` remains the historical hand-coded experiment.

The plotting cell uses trailing per-run means over 50 completed training episodes
and five development checkpoints. It averages runs' rolling means; the development
band is ±1 SD across smoothed runs, not a confidence interval. Set either window
in `highway_plot_windows` to `1` to inspect unsmoothed trends. Raw episode records
and both sets of rolling values are exported. Final scores use original test episodes.

Compare medium bins with SB3 DQN first, then use coarse/fine bins to inspect
discretization sensitivity. Unseen table keys have no learned entry; producing a
network output at a new observation does not establish reliable generalization.
Read reward, crashes and duration together. Equal interactions do not equal compute;
training-only timing excludes evaluation, and concurrent elapsed time depends on load.


### A.3.1 · Inspect policy behavior · Older DQN playback

This older supplied checkpoint used **20,000 interactions, seed 0** with the same
configuration. It is separate from the study above, which uses a different training
budget. Its original metadata and local re-evaluation remain in the assets.
Compare its behavior qualitatively; do not put it into the matched-budget ranking.
One successful animation does not establish reliable performance.


This playback runs only with `RUN_EXTRA_ANALYSIS=True`. It is not part of the core video.

This supplied PyTorch checkpoint and the historical hand-coded study are separate
from the SB3 experiment now used in the core lesson.

In [ ]:
if RUN_EXTRA_ANALYSIS:
    reference_agent = PlaybackDQN()
    reference_metadata = json.loads((ROOT / 'assets/checkpoints/dqn_reference_20k_seed0.json').read_text())
    print('Reference training budget:', reference_metadata['steps'], '| training seed:', reference_metadata['seed'])
    reference_rollout = highway_rollout(reference_agent, seed=2026,
                                        save_path=OUTPUT_DIR / 'reference_dqn.gif')
    print('Shown episode return:', reference_rollout.reward.sum(), '| Crash:', reference_rollout.crashed.any())
    display(Markdown('**Read the plot:** This supplied 20,000-step policy is a separate example, outside the matched-budget SB3 comparison.'))


### A.4 · Explore conflicting incentives · Prisoner's Dilemma

![Two players make simultaneous choices and receive separate payoffs.](../assets/figures/task_prisoners_dilemma.png)

**Optional self-study after the video.** Run setup and complete Task 1 first so
`rules` is available. Then run Sections A.4.1 and A.4.2 in order; no extra flag is
needed to load the supplied pair. Optional Task 4 is a separate practice exercise
in A.4.3.

**Why explore this alongside Simple Spread?** Chapter 3's single-agent loop treats the
other traffic as part of the environment. Here, each player's reward depends on
both players' choices, and we can let both players learn. Prisoner's Dilemma makes
**individual incentives** easy to see: taking the best immediate payoff for yourself
can leave both players worse off than mutual cooperation. Simple Spread used a
shared team score; here individual incentives can conflict. This is a mixed-incentive
game with both competitive pressure and possible mutual benefit.

#### A.4.1 · Recognize individual incentives · Payoffs and tit-for-tat

Each player chooses cooperate (`0`) or defect (`1`) without seeing the partner's
current action. The plot lists **(Player A, Player B)** payoffs. This is a general-sum
game: rewards are separate, not a shared team score and not necessarily opposites.
We use the unchanged bundled PettingZoo-Dilemmas environment.

Before learning, inspect a transparent comparison rule. **Tit-for-tat** cooperates
on the first round, then copies the partner's previous action. Marker `2` means
there was no previous action. It is hand-coded and never updates a Q-table.
Always cooperate and always defect are also fixed rules. These references help us
ask whether a learned policy behaves differently and how much its payoff depends
on its partner. The supplied function lets you compare behavior immediately. Optional Task 4
in A.4.3 provides a separate practice version.

In [ ]:
def tit_for_tat(previous_opponent_action):
    return 0 if previous_opponent_action == 2 else int(previous_opponent_action)

In [ ]:
check_tit_for_tat(tit_for_tat)
pd_matrix = plot_payoffs('prisoners_dilemma')
display(Markdown('**Read the plot:** Each cell gives (A, B): defecting against cooperation pays 5, mutual cooperation pays 3 each, and mutual defection pays 1 each.'))

#### A.4.2 · Reuse Q-learning for two players · Independent Q-learning

The payoff plot leaves a new question: what would players discover from experience?
A small state space lets us reuse the café's Q-learning idea. **Independent
Q-learning (IQL)** means each player keeps a separate Q-table, chooses an action
and updates from its own reward. We reuse the earlier learning rules in the provided
trainer; there is no extra required implementation task.

The state is `(rounds_remaining, own_previous_action, partner_previous_action)`.
Both choices use only past information and are submitted simultaneously. During
training, epsilon-greedy exploration samples actions. Because both players update,
each player's partner changes too: the transition and reward patterns it experiences
need not stay fixed. Independent learning does not guarantee convergence or cooperation.

The supplied pair was trained in self-play for **3,000 episodes of 20 rounds**,
seed 0: 60,000 joint decisions. Each has its own Q-table and receives its own payoff;
learning rate 0.15, discount 0.95, epsilon decays from 0.5 to 0.03. We load the final
checkpoint, turn exploration off (`epsilon=0`), and freeze updates (`learn=False`).
A greedy tie is broken by the agent's seeded RNG. These are illustrative policies
from one training run, not an algorithm ranking.

The table compares four choices in each role: three fixed rules and the trained
player for that role. Each entry is A's mean payoff over a **20-round evaluation**.
Read it alongside the mutual-cooperation table: a larger private payoff need not
mean more cooperation. Changing a selector starts a fresh match and does not train
or modify the saved Q-values. In Appendix A.6 you can train a fresh pair separately.


In [ ]:
pd_agent_a, pd_agent_b, pd_history, pd_metadata = load_social_reference(rules)
print(f"Independent Q-learning: {pd_metadata['episodes']:,} training episodes; separate Q-tables; evaluation updates OFF")
fixed_strategies = {'Always cooperate (fixed)': 'always_cooperate',
                    'Always defect (fixed)': 'always_defect', 'Tit-for-tat (fixed)': tit_for_tat}
strategies_a = {**fixed_strategies, 'Independent Q (trained A)': pd_agent_a}
strategies_b = {**fixed_strategies, 'Independent Q (trained B)': pd_agent_b}
tournament = []
for name, strategy in strategies_a.items():
    for opponent_name, opponent in strategies_b.items():
        match = social_match('prisoners_dilemma', strategy, opponent,
                             nrounds=20, epsilon=0., learn=False)
        tournament.append(dict(strategy=name, opponent=opponent_name,
                               mean_reward_A=match.reward_a.mean(),
                               mutual_cooperation=match.both_cooperate.mean()))
tournament = pd.DataFrame(tournament)
display(tournament.pivot(index='strategy', columns='opponent', values='mean_reward_A'))
display(Markdown('**Read the table:** Entries are A’s mean payoff over 20 frozen evaluation rounds; compare the learned row across partners, rather than declaring a universal winner.'))
display(tournament.pivot(index='strategy', columns='opponent', values='mutual_cooperation'))
display(Markdown('**Read the table:** Mutual cooperation is the fraction of rounds when both cooperate; it measures a different outcome from A’s private payoff.'))
if SHOW_WIDGETS:
    dilemma_ui = dilemma_controls(tit_for_tat, learned=(pd_agent_a, pd_agent_b))

**Explain:** Compare the learned row against always cooperate, always defect and
tit-for-tat. Does a learned player always earn more, or always cooperate? Why might
a Q-table trained against one learning partner behave differently against a fixed
partner? The evaluation freezes learning; the table does not show adaptation to
new partners. During training, both changing policies made the problem non-stationary.

#### A.4.3 · Optional Task 4: react to a partner · Tit-for-tat

Enable `RUN_OPTIONAL_PRACTICE` in setup. Fill the single `response` expression.
Action `0` means cooperate, `1` defect, and marker `2` means no previous action.
Cooperate initially, then copy the partner's previous action. The supplied
`tit_for_tat` function in A.4.1 remains available for comparison. This practice
function has its own name so an unfinished exercise cannot break the core flow.
Run the cell after filling it; expect **“Task 4 passed”**.


In [ ]:
if RUN_OPTIONAL_PRACTICE:
    def practice_tit_for_tat(previous_opponent_action):
        response = ...  # TODO: cooperate initially; otherwise copy the previous action.
        if response is Ellipsis:
            raise NotImplementedError('Optional Task 4: fill response.')
        return int(response)
    check_tit_for_tat(practice_tit_for_tat)


### A.5 · Optional Task 5: calculate a shared reward

With `RUN_OPTIONAL_PRACTICE=True`, fill the `reward` expression below. The provided
broadcasting creates a distance matrix with shape `(n_agents, n_landmarks)`.
For **each landmark**, find its nearest agent, sum those distances, then negate.
Choose the reduction axis carefully: rows represent agents, columns represent landmarks.
Return one scalar reward, not a vector of agent scores.

With landmarks at `(0,0)` and `(10,0)`, agents at `(1,0)` and `(2,0)` give reward −9.
Putting both agents at the left landmark gives −10; one on each landmark gives 0.
Run the completed cell; expect **“Task 5 passed”**.


In [ ]:
if RUN_OPTIONAL_PRACTICE:
    def practice_team_reward(agent_positions, landmark_positions):
        distances = np.linalg.norm(
            agent_positions[:, None, :] - landmark_positions[None, :, :], axis=-1)
        reward = ...  # TODO: minimum over agents, sum over landmarks, then negate.
        if reward is Ellipsis:
            raise NotImplementedError('Optional Task 5: fill the reward expression.')
        return float(reward)
    check_team_reward(practice_team_reward)


### A.6 · Train a fresh pair · Independent Q-learning

Appendix A.4.2 loads trained Q-tables and freezes them for evaluation. Enable
`RUN_SOCIAL_TRAINING` to train a new pair with the current mode's `social` budget
and the learning rules you completed earlier. This separate run does not replace
the supplied checkpoint or the appendix tournament. The state contains remaining
rounds and previous joint actions; each agent updates from its own payoff.
Two changing policies make each other's learning problem non-stationary.
Convergence and cooperation are not guaranteed. The adapter scores exactly 20
rounds and ignores the upstream version's extra empty completion call.

MAPPO training is kept in `scripts/run_multi_agent.py` so it does not run accidentally
while recording. That runner calls the installed library algorithm; it does not
supply a custom PPO update. See README for reproduction commands and provenance.

**MAPPO preparation reference (outside the recording):** All three agents share
one actor and centralized critic. The actor appends six locally derived distances
to its 18-value observation (24 model inputs); the critic derives the same features
from all three 18-value observations in the global state (72 model inputs from 54
raw values). No target assignments or expert actions enter training. LayerNorm,
two 64-unit ReLU hidden layers, and the library's PPO updates fit the model.

Feature and hidden-layer normalization help fit the networks. These adapter details
are provided; they are not another student implementation task.

The simulator's 60-step cutoff is a truncation. Training uses the library's
**time-limit bootstrapping**, following the same termination/cutoff distinction as
Task 2. A positive constant factor of 0.1 scales reward internally for the critic;
it preserves the reward objective. All printed returns and plots use the original
unscaled coverage reward. The skrl 1.4.3 adapter uses a wide value-clipping window
to obtain an unclipped critic despite that release's per-agent flag handling.
The reward formula, actions, physics and evaluation horizon remain unchanged.

Training uses 80% random starts and 20% the practiced clustered scene. The selected checkpoint
is chosen on 16 development starts; 50 disjoint random starts are reserved until
that choice is fixed. Candidate settings and curves are recorded alongside the
selected checkpoint. See README for training, resumption and final-evaluation commands.


In [ ]:
if RUN_SOCIAL_TRAINING:
    fresh_a, fresh_b, fresh_history = train_social('prisoners_dilemma', rules,
        episodes=budget['social'], opponent='learner', seed=0)
    fresh_match = social_match('prisoners_dilemma', fresh_a, fresh_b, learn=False)
    plot_social(fresh_history, fresh_match, 'prisoners_dilemma')
    display(Markdown('**Read the plots:** The curve tracks mutual cooperation during this fresh training run; the heatmap shows joint choices in one frozen greedy evaluation session.'))
else:
    print('Optional: enable RUN_SOCIAL_TRAINING for a fresh pair; the main demo already loaded trained Independent Q policies.')

### A.7 · Keep experiments reproducible · Save results

Results go to a new directory under `results/`, which Git ignores. The run manifest labels
fresh training versus cached comparisons. Figures and GIFs can be reused in your notes.
No Google account, Drive mount, external dataset, or GPU is required for the local lab.

Prisoner's Dilemma results are included only if you ran Appendix A.4.2.

In [ ]:
np.savez_compressed(OUTPUT_DIR / 'cafe_q_table.npz',q=cafe_agent.q,visits=cafe_agent.visits)
cafe_history.to_csv(OUTPUT_DIR / 'cafe_history.csv',index=False)
cafe_scoreboard.to_csv(OUTPUT_DIR / 'cafe_scoreboard.csv')
if globals().get('cafe_policy_comparison') is not None:
    cafe_policy_comparison.to_csv(OUTPUT_DIR / 'cafe_policy_comparison.csv', index=False)
save_study(highway_study, OUTPUT_DIR / 'highway_study')
highway_training_smoothed.to_csv(OUTPUT_DIR / 'highway_training_rolling_mean.csv', index=False)
highway_evaluation_smoothed.to_csv(OUTPUT_DIR / 'highway_evaluation_rolling_mean.csv', index=False)
if 'tournament' in globals():  # Optional Appendix A.4 may be skipped.
    tournament.to_csv(OUTPUT_DIR / 'prisoners_dilemma_tournament.csv',index=False)
spread_scoreboard.to_csv(OUTPUT_DIR / 'simple_spread_scoreboard.csv',index=False)
nearest_rollout['history'].to_csv(OUTPUT_DIR / 'simple_spread_nearest.csv',index=False)
mappo_rollout['history'].to_csv(OUTPUT_DIR / 'simple_spread_mappo.csv',index=False)
spread_test_results.to_csv(OUTPUT_DIR / 'simple_spread_heldout_evaluation.csv',index=False)
assigned_rollout['history'].to_csv(OUTPUT_DIR / 'simple_spread_assignment.csv',index=False)
shown_dqn.save(OUTPUT_DIR / 'shown_sb3_dqn.zip')
manifest = {'run_mode':RUN_MODE,'versions':VERSIONS,'budget':budget,
            'cafe_source':'trained in this run',
            'library_agent':{'package':'stable-baselines3','version':VERSIONS['stable-baselines3'],
                             'api_demo_trained':RUN_SB3_TRAINING,'api_demo_steps':library_agent.num_timesteps,
                             'experiment_implementation':highway_study.metadata['dqn_implementation']},
            'cafe_game':CookieCafe.GAME_ID,'cafe_product':CookieCafe.PRODUCT_NAME,
            'cafe_state':['hour','stock'],
            'cafe_demand_means':CookieCafe.MEANS.tolist(),
            'cafe_selling_hours':list(range(10, 10 + len(CookieCafe.MEANS))),
            'cafe_uncertainty':'Poisson demand only; preparation always succeeds',
            'cafe_reference':{'method':'known-model backward induction',
                              'future_customers_known':False,
                              'expected_daily_reward':cafe_reference.expected_opening_return},'highway_comparison_source':highway_source,
            'highway_config':HIGHWAY_CONFIG,'highway_study':highway_study.metadata,
            'highway_plot_windows':highway_plot_windows,'reference_rollout':'supplied 20,000-step checkpoint, seed 0' if RUN_EXTRA_ANALYSIS else 'not run (optional)',
            'prisoners_dilemma':globals().get('pd_metadata'),
            'fresh_optional_iql_training':RUN_SOCIAL_TRAINING,
            'simple_spread':{'package':'mpe2','version':VERSIONS['mpe2'],'local_ratio':0.0,
                             'agents':3,'steps':60,'seed':2026,'layout':'clustered',
                             'controllers':['nearest (fixed)','MAPPO (skrl, learned)','assignment (fixed, centralized)'],
                             'training_in_notebook':False,'mappo':mappo_metadata}}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(manifest,indent=2))
print('Saved results:',OUTPUT_DIR)


### A.8 · Explore further · References

**References:** [Gymnasium environment API](https://gymnasium.farama.org/api/env/),
[time limits](https://gymnasium.farama.org/tutorials/gymnasium_basics/handling_time_limits/),
[HighwayEnv](https://highway-env.farama.org/),
[PyTorch DQN tutorial](https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html),
[PettingZoo-Dilemmas source](https://github.com/mrochk/pettingzoo-dilemmas),
[Simple Spread reward definition](https://mpe2.farama.org/environments/simple_spread/).
This lab pins MPE2 0.0.1 for compatibility with the existing pygame-based Mac environment.

Lecture connections from the supplied notebook: `3.1_RLmodels-part1`, pp. 30–35 and 46–53;
`3.2_DeepRL-part1`, pp. 15–27; `3.2_DeepRL-part2`, pp. 32–39.
The lecture PDFs are not redistributed here. See `vendor/PROVENANCE.json` for the pinned game commit.

[Stable-Baselines3 DQN](https://stable-baselines3.readthedocs.io/en/v2.7.1/modules/dqn.html).

[skrl MAPPO, pinned implementation](https://github.com/Toni-SM/skrl/blob/1.4.3/skrl/multi_agents/torch/mappo/mappo.py),
[MAPPO paper](https://arxiv.org/abs/2103.01955).
